# Data Courtroom — investigação com Spark

**Estudo posterior · sua conta OCI · caso fictício · no seu ritmo**

Importe este `.ipynb` no seu workspace AIDP e associe um compute Spark. Execute as seções na ordem e confira os resultados antes de avançar. O notebook contém os dados fictícios; não usa OCIDs, chaves ou caminhos da conta da apresentadora.

O processamento acontece no compute que você associar. Não grava tabelas persistentes, não cria agentes e não chama modelos. Essa parte prática cobre preparação, consulta e revisão de documentos. Os agentes da apresentação são uma integração separada, descrita no material.

Para começar sem compute cloud, use `laboratorio_local.py`: é uma prática local explicitamente separada do AIDP, não uma chamada cloud.


## 0. Diagnóstico — comece por aqui

Confirme que há uma sessão Spark disponível. Se esta célula falhar, o ambiente ainda não está pronto para a prática.

In [ ]:
try:
    spark
except NameError as exc:
    raise RuntimeError("Associe este notebook a um compute Spark do AIDP antes de continuar.") from exc
print("Spark:", spark.version)
print("Linhas de diagnóstico:", spark.range(3).count())


## 1. Material do caso

A célula abaixo descompacta o mesmo CSV fictício da demo. O conteúdo está embutido para dispensar download externo e acesso à conta da apresentadora. Você pode expandi-la para inspecionar. Não há credenciais.

In [ ]:
import base64, zlib, csv, io
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
DATA_B64 = '...'
reader = csv.DictReader(io.StringIO(zlib.decompress(base64.b64decode(DATA_B64)).decode("utf-8")))
columns = reader.fieldnames
rows = [[r[c] if r[c] != "" else None for c in columns] for r in reader]
schema = StructType([StructField(c, StringType(), True) for c in columns])
bronze = spark.createDataFrame(rows, schema).withColumn("valor", F.col("valor").cast("decimal(12,2)"))
print("Registros recebidos:", bronze.count())
bronze.select("tentativa_id", "dia", "metodo", "valor").show(5)


## 2. Investigar a qualidade

Antes de executar: qual é a diferença entre uma linha recebida e uma tentativa válida? Leia as regras abaixo. Identifique uma regra que você revisaria com alguém do negócio.

Trabalhamos apenas com DataFrames desta sessão; não regravamos as tabelas da apresentação.


In [ ]:
validos = bronze.filter(
    F.col("tentativa_id").isNotNull()
    & (F.col("valor") > 0)
    & F.col("metodo").isin("cartao", "pix")
)
silver = validos.dropDuplicates(["tentativa_id"])
print({"recebidos": bronze.count(), "invalidos": bronze.count() - validos.count(),
       "duplicatas": validos.count() - silver.count(), "validos": silver.count()})


### Experimento sem alterar a origem

Compare o total somado antes e depois da deduplicação. A diferença que aparecer é efeito da preparação, não uma inferência de um modelo.


In [ ]:
validos.agg(F.sum("valor").alias("total_antes_deduplicacao")).show()
silver.agg(F.sum("valor").alias("total_apos_deduplicacao")).show()


## 3. Interrogar os números

Agora compare dia e método. Primeiro observe; depois escreva uma frase que descreva o que mudou e outra sobre o que os números, sozinhos, não demonstram.

Esta consulta usa sua cópia em memória no Spark. Na apresentação, Byte usa uma ferramenta SQL sobre a tabela externa da apresentadora. São percursos diferentes do mesmo conjunto fictício.


In [ ]:
gold = silver.groupBy("dia", "metodo").agg(
    F.count("*").alias("tentativas"),
    F.sum(F.when(F.col("status") == "aprovado", 1).otherwise(0)).alias("aprovados"),
    F.sum(F.when(F.col("status") == "aprovado", F.col("valor")).otherwise(0)).alias("receita")
).orderBy("dia", "metodo")
gold.show(truncate=False)


### Uma pergunta diferente

Escolha um método em `metodo_escolhido` e compare a taxa de aprovação. Não confunda mudança em pontos percentuais com redução percentual de receita.


In [ ]:
metodo_escolhido = "cartao"  # Experimente "pix" depois.
gold.filter(F.col("metodo") == metodo_escolhido).withColumn(
    "taxa_aprovacao_percentual", F.round(F.col("aprovados") / F.col("tentativas") * 100, 1)
).show(truncate=False)


## 4. Conferir as fontes

Esta é leitura direta de documentos do kit; não é RAG. Compare o que você lê com a resposta de Íris na apresentação.

Anote: documento, horário ou período, trecho relevante e o que ele **não** permite concluir. Registre sua hipótese antes de ler um parecer.


In [ ]:
DOCUMENTOS = {'campanha.txt': 'CASO FICTÍCIO AURORA — material didático\nEvidência: E04\nTítulo: Calendário da campanha de aquisição\nVersão: 1.0\nData: 13/09/2026\nLocalização: Vigência\nFuso do caso: America/Sao_Paulo\n\nA alteração da campanha entrou em vigor em 13/09/2026. O incidente investigado é de 20/09/2026. Nos dois dias comparados, há 500 tentativas no canal app e 500 no canal web. Não há mudança de volume de tentativas que explique a queda de aprovações no cartão.\n', 'mudanca-pagamento.txt': 'CASO FICTÍCIO AURORA — material didático\nEvidência: E02\nTítulo: Registro de alteração e log de pagamento\nVersão: 1.0\nData: 20/09/2026\nLocalização: Seções 1–2\nFuso do caso: America/Sao_Paulo\n\n10h00 — A configuração do fluxo de cartão passou da versão cfg-16 para cfg-17. A alteração atingiu o parâmetro de validação de credencial do gateway.\n10h07 — Exemplo de registro: método=cartao; status=falhou; erro=CONFIG_INVALID; config=cfg-17. Os registros adicionais de falha por configuração do caso aparecem após 10h00.\nA equipe deve comparar o comportamento anterior e validar a configuração antes de recomendar reversão.\n', 'reconciliacao.txt': 'CASO FICTÍCIO AURORA — material didático\nEvidência: E03\nTítulo: Reconciliação de pedidos e pagamentos\nVersão: 1.0\nData: 20/09/2026\nLocalização: Resumo da carga\nFuso do caso: America/Sao_Paulo\n\nApós excluir duas duplicatas e um registro inválido de transporte, os arquivos contêm 1.000 tentativas válidas no dia 19/09 e 1.000 no dia 20/09. Cada tentativa possui uma referência de pedido e uma tentativa de pagamento. As contagens conferem com os manifests fictícios de origem. Não há atraso de carga no recorte do caso.\n'}


In [ ]:
documento_escolhido = "mudanca-pagamento.txt"  # Depois leia reconciliacao.txt e campanha.txt.
print(DOCUMENTOS[documento_escolhido])


## 5. Seu registro de investigação

Preencha antes de consultar a conclusão de outra pessoa ou de um agente. Não há gabarito de causa neste notebook.

- Observação numérica:
- Documento e trecho:
- Hipótese que eu levaria ao júri:
- Limitação das provas:
- Próxima verificação:

Ao revisar um parecer, compare a linguagem: ele descreveu uma hipótese ou afirmou uma causa comprovada?


## Encerramento

Este notebook não criou tabelas persistentes. Encerre sua sessão e confira a política de parada do seu compute para evitar uso desnecessário. Recursos de infraestrutura criados separadamente continuam na sua conta até serem removidos por você.
